In [0]:
# ============================================================
# FASE 1 — Extract: Cine italiano desde TMDB API
# ============================================================
import requests
from pyspark.sql.types import (
    StructType, StructField, StringType,
    DoubleType, IntegerType
)
from pyspark.sql import Row
from datetime import datetime

# ----------------------------------------------------------
# 1. Recuperar credenciales desde Databricks Secrets
# ----------------------------------------------------------
api_key = dbutils.secrets.get(scope="tmdb", key="api_key")
print(f"API key cargada: {api_key}")  # → [REDACTED]

# ----------------------------------------------------------
# 2. Parámetros de búsqueda
# ----------------------------------------------------------
BASE_URL    = "https://api.themoviedb.org/3"
LANGUAGE    = "it"       # idioma original: italiano
REGION      = "IT"       # región Italia
NUM_PAGES   = 10         # páginas a traer (20 películas/página = 200 películas)

# ----------------------------------------------------------
# 3. Llamada paginada al endpoint discover/movie
# ----------------------------------------------------------
peliculas_raw = []

for page in range(1, NUM_PAGES + 1):
    response = requests.get(
        f"{BASE_URL}/discover/movie",
        params={
            "api_key":               api_key,
            "with_original_language": "it",   # solo cine en italiano original
            "sort_by":               "popularity.desc",
            "page":                  page,
            "include_adult":         False,
        }
    )

    if response.status_code != 200:
        print(f"Error en página {page}: {response.status_code}")
        break

    data = response.json()
    peliculas_raw.extend(data.get("results", []))
    print(f"Página {page}/{NUM_PAGES} — {len(data.get('results', []))} películas obtenidas")

print(f"\nTotal películas extraídas: {len(peliculas_raw)}")

# ----------------------------------------------------------
# 4. Parsear resultados → filas planas
# ----------------------------------------------------------
filas = []
for p in peliculas_raw:
    filas.append(Row(
        pelicula_id      = str(p.get("id", "")),
        titulo           = p.get("title", ""),
        titulo_original  = p.get("original_title", ""),
        fecha_estreno    = p.get("release_date", ""),
        anio_estreno     = int(p["release_date"][:4]) if p.get("release_date") else None,
        popularidad      = float(p.get("popularity", 0.0)),
        valoracion_media = float(p.get("vote_average", 0.0)),
        num_votos        = int(p.get("vote_count", 0)),
        idioma_original  = p.get("original_language", ""),
        descripcion      = p.get("overview", ""),
        genero_ids       = ",".join(str(g) for g in p.get("genre_ids", [])),
    ))

# ----------------------------------------------------------
# 5. Schema explícito y creación del DataFrame
# ----------------------------------------------------------
peliculas_schema = StructType([
    StructField("pelicula_id",      StringType(),  False),
    StructField("titulo",           StringType(),  True),
    StructField("titulo_original",  StringType(),  True),
    StructField("fecha_estreno",    StringType(),  True),
    StructField("anio_estreno",     IntegerType(), True),
    StructField("popularidad",      DoubleType(),  True),
    StructField("valoracion_media", DoubleType(),  True),
    StructField("num_votos",        IntegerType(), True),
    StructField("idioma_original",  StringType(),  True),
    StructField("descripcion",      StringType(),  True),
    StructField("genero_ids",       StringType(),  True),
])

df_peliculas_raw = spark.createDataFrame(filas, schema=peliculas_schema)

df_peliculas_raw.show(5, truncate=False)
df_peliculas_raw.printSchema()
print(f"Total filas en DataFrame: {df_peliculas_raw.count()}")

# ----------------------------------------------------------
# 6. Guardar en Unity Catalog (tabla Delta gestionada)
# ----------------------------------------------------------
fecha_ingesta = datetime.now().strftime("%Y%m%d")

# Crear la base de datos si no existe
spark.sql("CREATE DATABASE IF NOT EXISTS etl_cine")

# Guardar como tabla Delta gestionada por Unity Catalog
df_peliculas_raw.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("etl_cine.peliculas_italianas_raw")

print(f"✓ Datos guardados en tabla: etl_cine.peliculas_italianas_raw")
print(f"✓ Fecha ingesta: {fecha_ingesta}")

In [0]:
spark.sql("SELECT COUNT(*), MIN(anio_estreno), MAX(anio_estreno) FROM etl_cine.peliculas_italianas_raw").show()